# Brain Tumor Detector — Multi-class MRI Classification  *(High-Accuracy variant)*

> **Experimental copy.** Deeper fine-tuning (top 120 layers, 35 epochs) to chase the last ~0.3% accuracy. Higher overfit risk to the test set — compare against the baseline copy and keep whichever generalises better. EarlyStopping still applies.

Classify each brain MRI scan into one of **4 classes**: `glioma`, `meningioma`, `notumor`, `pituitary`.

**Engineering protocol (what makes this rigorous, not just "I got 99%"):**

1. **Data split** — Training folder split 80 / 20 into train / validation (`seed=42`). The original **Testing** folder is the held-out test set.
2. **Golden rule** — *no decision* (model choice, epochs, learning rate, fine-tuning) is ever made by looking at the test set. The test set is touched **exactly once**, at the very end.
3. **Selection metric = Macro-F1** (balances precision & recall, not gameable). **Clinical analysis metric = per-class Recall** (sensitivity — how many tumors we miss).
4. **Pipeline:** Baseline CNN → CNN + augmentation → compare ResNet50 / EfficientNetB0 / DenseNet121 (frozen) on validation → pick winner by Macro-F1 → fine-tune (BatchNorm kept frozen) → single test evaluation → Grad-CAM → limitations & future work.

> **Runtime:** set Colab to a **GPU** runtime (`Runtime → Change runtime type → GPU`).

## 1. Setup & Data Download

In [ ]:
!pip install --upgrade --no-cache-dir gdown -q

In [ ]:
# Download the dataset zip (same source as the helper notebook) and unzip it.
# NOTE: recent gdown removed the `--id` flag, so we use the positional form with a
# fuzzy-URL fallback. If both fail (dead/over-quota Drive link), use the Kaggle
# fallback in the next cell instead.
!gdown 1bXBSfKDaItFigHa5QfcnyTXADG2wlWJj || gdown --fuzzy "https://drive.google.com/uc?id=1bXBSfKDaItFigHa5QfcnyTXADG2wlWJj"
!unzip -q -o brain_tumor.zip
!ls

**Fallback (only if the gdown cell above failed).** Upload your `kaggle.json`, then run:
```python
!pip install kaggle -q
from google.colab import files; files.upload()          # pick kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset
!unzip -q -o brain-tumor-mri-dataset.zip
```

## 2. Imports, Config & Reproducibility

In [ ]:
import os, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import (confusion_matrix, classification_report, f1_score,
                             recall_score, precision_score, accuracy_score,
                             roc_auc_score, roc_curve, auc)

print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

# --- Reproducibility -------------------------------------------------------
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# --- Config ---------------------------------------------------------------
IMG_DIMS    = (128, 128)
IMG_SHAPE   = IMG_DIMS + (3,)
BATCH_SIZE  = 32
CLASSES     = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES = len(CLASSES)
AUTOTUNE    = tf.data.AUTOTUNE

# --- Robust dataset path detection ----------------------------------------
# Finds the Training/Testing folders wherever the zip extracted them (handles
# nested folders), so the notebook doesn't break on './Training' not existing.
def _find_dir(targets):
    targets = [t.lower() for t in targets]
    for root, dirs, _ in os.walk('.'):
        for d in dirs:
            if d.lower() in targets:
                p = os.path.join(root, d)
                try:
                    subs = [s.lower() for s in os.listdir(p)]
                except OSError:
                    continue
                if any(c in subs for c in CLASSES):
                    return p
    return None

TRAIN_DIR = _find_dir(['training', 'train']) or './Training'
TEST_DIR  = _find_dir(['testing', 'test'])   or './Testing'

def count_per_class(directory):
    return {c: len(os.listdir(os.path.join(directory, c))) for c in CLASSES}

print('TRAIN_DIR =', TRAIN_DIR, '|  TEST_DIR =', TEST_DIR)
assert TRAIN_DIR and os.path.isdir(TRAIN_DIR) and os.path.isdir(TEST_DIR), \
    'Dataset folders not found — the download/unzip step did not complete. Re-run section 1.'

## 3. Data Loading — Train / Validation / Test

- **Train**: 80% of `Training/`
- **Validation**: 20% of `Training/` (used for *all* model-selection decisions)
- **Test**: the original `Testing/` folder, evaluated only once at the end

In [ ]:
def prepare_train_and_val_datasets():
    train_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, validation_split=0.2, subset='training',
        class_names=CLASSES, seed=SEED, image_size=IMG_DIMS, batch_size=BATCH_SIZE)
    val_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, validation_split=0.2, subset='validation',
        class_names=CLASSES, seed=SEED, image_size=IMG_DIMS, batch_size=BATCH_SIZE)
    return train_ds, val_ds

def prepare_test_dataset():
    # shuffle=False so labels stay aligned and the evaluation is deterministic
    return tf.keras.utils.image_dataset_from_directory(
        TEST_DIR, class_names=CLASSES, seed=SEED, image_size=IMG_DIMS,
        batch_size=BATCH_SIZE, shuffle=False)

train_ds, val_ds = prepare_train_and_val_datasets()
test_ds = prepare_test_dataset()

# Labels are integers -> use sparse categorical crossentropy throughout.
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)

In [ ]:
# Class balance check -> justifies NOT using class weights (classes are ~balanced).
print('Train class counts:', count_per_class(TRAIN_DIR))

In [ ]:
# Visualise a few training images
plt.figure(figsize=(12, 6))
for images, labels in train_ds.take(1):
    for i in range(8):
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(CLASSES[int(labels[i])]); plt.axis('off')
plt.tight_layout(); plt.show()

## 4. Experiment Logging & Evaluation Helpers

Every experiment appends a row to `experiment_log`. Evaluation reports
**accuracy, macro-precision, macro-recall, macro-F1, per-class recall and macro-AUC** —
not accuracy alone.

In [ ]:
experiment_log = []

def log_experiment(name, val_metrics, n_params, train_time, notes=''):
    rec = {
        'Model'           : name,
        'Val Accuracy'    : round(val_metrics['accuracy'], 4),
        'Val Macro-F1'    : round(val_metrics['macro_f1'], 4),
        'Val Macro Recall': round(val_metrics['macro_recall'], 4),
        'Val Macro AUC'   : round(val_metrics['macro_auc'], 4),
        'Glioma Recall'   : round(val_metrics['per_class_recall']['glioma'], 4),
        'Params'          : int(n_params),
        'Train Time (s)'  : round(train_time, 1),
        'Notes'           : notes,
    }
    experiment_log.append(rec)
    return pd.DataFrame(experiment_log)

def plot_history(history, title=''):
    h = history.history
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    ax[0].plot(h['accuracy'], label='train')
    ax[0].plot(h.get('val_accuracy', []), label='val')
    ax[0].set_title(f'{title} — Accuracy'); ax[0].set_xlabel('epoch'); ax[0].legend()
    ax[1].plot(h['loss'], label='train')
    ax[1].plot(h.get('val_loss', []), label='val')
    ax[1].set_title(f'{title} — Loss'); ax[1].set_xlabel('epoch'); ax[1].legend()
    plt.tight_layout(); plt.show()

In [ ]:
def get_predictions(model, ds):
    y_true, y_prob = [], []
    for imgs, labels in ds:
        y_prob.append(model.predict(imgs, verbose=0))
        y_true.append(labels.numpy())
    y_true = np.concatenate(y_true)
    y_prob = np.concatenate(y_prob)
    y_pred = np.argmax(y_prob, axis=1)
    return y_true, y_pred, y_prob

def compute_metrics(y_true, y_pred, y_prob):
    per_class = recall_score(y_true, y_pred, average=None, labels=range(NUM_CLASSES))
    m = {
        'accuracy'       : accuracy_score(y_true, y_pred),
        'macro_f1'       : f1_score(y_true, y_pred, average='macro'),
        'macro_recall'   : recall_score(y_true, y_pred, average='macro'),
        'macro_precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'per_class_recall': {CLASSES[i]: per_class[i] for i in range(NUM_CLASSES)},
    }
    try:
        m['macro_auc'] = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
    except Exception:
        m['macro_auc'] = float('nan')
    return m

def evaluate_on_validation(model):
    return compute_metrics(*get_predictions(model, val_ds))

def plot_confusion_matrix(cm, title='Confusion Matrix'):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASSES, rotation=45, ha='right')
    ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASSES)
    thresh = cm.max() / 2.0
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, cm[i, j], ha='center',
                    color='white' if cm[i, j] > thresh else 'black')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title)
    plt.colorbar(im); plt.tight_layout(); plt.show()

def plot_roc(y_true, y_prob, title='ROC (one-vs-rest)'):
    y_bin = tf.keras.utils.to_categorical(y_true, NUM_CLASSES)
    plt.figure(figsize=(7, 6))
    for i in range(NUM_CLASSES):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        plt.plot(fpr, tpr, label=f'{CLASSES[i]} (AUC={auc(fpr, tpr):.3f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title(title); plt.legend(); plt.tight_layout(); plt.show()

def full_report(model, ds, title='Test'):
    y_true, y_pred, y_prob = get_predictions(model, ds)
    m = compute_metrics(y_true, y_pred, y_prob)
    print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))
    print(f"Macro-F1     : {m['macro_f1']:.4f}")
    print(f"Macro Recall : {m['macro_recall']:.4f}")
    print(f"Macro AUC    : {m['macro_auc']:.4f}")
    plot_confusion_matrix(confusion_matrix(y_true, y_pred), f'{title} — Confusion Matrix')
    return m, y_true, y_pred, y_prob

def make_callbacks(filepath, patience=5):
    return [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience,
                                         restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                             patience=2, min_lr=1e-6, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(filepath, monitor='val_loss',
                                           save_best_only=True),
    ]

## 5. Experiment 1 — Baseline CNN (from scratch)

A simple 3-conv-block network. This is the **reference point**; everything else is
measured against it. Expected ~90–93%.

In [ ]:
def build_baseline_cnn():
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=IMG_SHAPE),
        tf.keras.layers.Rescaling(1. / 255),
        tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),
    ], name='Baseline_CNN')

baseline = build_baseline_cnn()
baseline.compile(optimizer='adam', loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])
baseline.summary()

t0 = time.time()
hist_baseline = baseline.fit(train_ds, validation_data=val_ds, epochs=20,
                             callbacks=make_callbacks('baseline.keras'))
t_baseline = time.time() - t0

plot_history(hist_baseline, 'Baseline CNN')
m = evaluate_on_validation(baseline)
log_experiment('Baseline CNN', m, baseline.count_params(), t_baseline,
               'From scratch, 3 conv blocks')

## 6. Experiment 2 — Deeper CNN + BatchNorm + Dropout + Augmentation

Augmentation is added as **layers inside the model**, so it is automatically
disabled at inference. MRI-safe transforms only (gentle flip / rotation / zoom /
contrast — no aggressive rotation or shear that would distort anatomy).

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name='data_augmentation')

def conv_bn_block(x, filters):
    x = tf.keras.layers.Conv2D(filters, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    return tf.keras.layers.MaxPooling2D()(x)

def build_cnn_v2():
    inputs = tf.keras.Input(shape=IMG_SHAPE)
    x = data_augmentation(inputs)
    x = tf.keras.layers.Rescaling(1. / 255)(x)
    x = conv_bn_block(x, 32)
    x = conv_bn_block(x, 64)
    x = conv_bn_block(x, 128)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs, name='CNN_v2_Aug')

cnn_v2 = build_cnn_v2()
cnn_v2.compile(optimizer='adam', loss='sparse_categorical_crossentropy',
               metrics=['accuracy'])

t0 = time.time()
hist_v2 = cnn_v2.fit(train_ds, validation_data=val_ds, epochs=30,
                     callbacks=make_callbacks('cnn_v2.keras'))
t_v2 = time.time() - t0

plot_history(hist_v2, 'CNN v2 + Augmentation')
m = evaluate_on_validation(cnn_v2)
log_experiment('CNN + Augmentation', m, cnn_v2.count_params(), t_v2,
               'BatchNorm + Dropout + augmentation')

## 7. Transfer Learning — ResNet50 / EfficientNetB0 / DenseNet121

**Critical detail:** each backbone uses its *own* `preprocess_input`. Using a single
`Rescaling(1/255)` for all of them would silently hurt accuracy — especially for
EfficientNet, which expects raw [0, 255] inputs and normalises internally.

The backbone is called with `training=False`, which keeps its BatchNorm layers in
inference mode (important once we fine-tune).

In [ ]:
TRANSFER_BACKBONES = {
    'ResNet50':       (tf.keras.applications.ResNet50,
                       tf.keras.applications.resnet.preprocess_input),
    'EfficientNetB0': (tf.keras.applications.EfficientNetB0,
                       tf.keras.applications.efficientnet.preprocess_input),
    'DenseNet121':    (tf.keras.applications.DenseNet121,
                       tf.keras.applications.densenet.preprocess_input),
}

def build_transfer_model(name):
    backbone_fn, preprocess_fn = TRANSFER_BACKBONES[name]
    base = backbone_fn(include_top=False, weights='imagenet', input_shape=IMG_SHAPE)
    base.trainable = False
    inputs = tf.keras.Input(shape=IMG_SHAPE)
    x = data_augmentation(inputs)
    x = preprocess_fn(x)                 # <-- per-model preprocessing
    x = base(x, training=False)          # <-- BN stays in inference mode
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs, name=name), base

transfer_models = {}

def run_transfer(name, epochs=15):
    model, base = build_transfer_model(name)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    t0 = time.time()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs,
                     callbacks=make_callbacks(f'{name}.keras'))
    t = time.time() - t0
    plot_history(hist, name)
    m = evaluate_on_validation(model)
    log_experiment(name, m, model.count_params(), t, 'Transfer learning (frozen backbone)')
    transfer_models[name] = (model, base)
    return model, base

In [ ]:
run_transfer('ResNet50')

In [ ]:
run_transfer('EfficientNetB0')

In [ ]:
run_transfer('DenseNet121')

## 8. Validation Comparison — Frozen Backbones

Leaderboard so far (frozen backbones). This is **informational only** — the
frozen-stage ranking does *not* reliably predict the ranking after fine-tuning,
so we fine-tune all three backbones in the next step before picking a final model.

In [ ]:
frozen_df = (pd.DataFrame(experiment_log)
             .sort_values('Val Macro-F1', ascending=False)
             .reset_index(drop=True))
display(frozen_df)

## 9. Fine-Tuning (all three backbones)

We unfreeze the top layers of each backbone with a **low learning rate (5e-5)**.
**BatchNorm layers are kept frozen** — updating their running statistics during
fine-tuning is a classic bug that quietly destroys accuracy. (The `training=False`
call above already enforces inference-mode BN; we also set `trainable=False` on BN
layers for clarity.)

We fine-tune **all three** backbones because the frozen ranking is not a reliable
predictor of the fine-tuned ranking (e.g. ResNet50 with caffe-style preprocessing
and frozen BN often trails on small 128x128 inputs, then DenseNet121 / EfficientNetB0
overtake it after fine-tuning).

In [ ]:
def fine_tune(model, base, unfreeze_last=120, lr=5e-5, epochs=35):
    base.trainable = True
    # freeze all but the top `unfreeze_last` layers, and ALWAYS freeze BatchNorm
    for layer in base.layers[:-unfreeze_last]:
        layer.trainable = False
    for layer in base.layers[-unfreeze_last:]:
        layer.trainable = not isinstance(layer, tf.keras.layers.BatchNormalization)

    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    t0 = time.time()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs,
                     callbacks=make_callbacks(f'{model.name}_finetuned.keras', patience=6))
    t = time.time() - t0
    plot_history(hist, f'{model.name} (fine-tuned)')
    m = evaluate_on_validation(model)
    log_experiment(f'{model.name} (fine-tuned)', m, model.count_params(), t,
                   f'Top {unfreeze_last} layers unfrozen, BN frozen, lr={lr}')
    return model

# Fine-tune every transfer backbone. (Lower `epochs` here if you are short on time.)
finetuned = {}
for name in TRANSFER_BACKBONES:
    model_i, base_i = transfer_models[name]
    fine_tune(model_i, base_i)
    finetuned[name] = (model_i, base_i)

### 9b. Final Model Selection (from the fine-tuned models)

The final model is the fine-tuned backbone with the highest **validation Macro-F1**.
Still chosen without ever touching the test set.

In [ ]:
ft_log = pd.DataFrame(experiment_log)
ft_df = (ft_log[ft_log['Model'].str.contains('fine-tuned')]
         .sort_values('Val Macro-F1', ascending=False)
         .reset_index(drop=True))
display(ft_df)

best_name = ft_df.iloc[0]['Model'].replace(' (fine-tuned)', '')
best_model, best_base = finetuned[best_name]
print(f'Final selected model (highest fine-tuned Val Macro-F1): {best_name}')

### 9c. Ensemble + TTA + Bootstrap — helpers and selection

Three extra rigor/accuracy upgrades that reuse the **already-trained** fine-tuned
models (no extra training):

- **Ensemble** — average the softmax probabilities of all three fine-tuned
  backbones. Whether the ensemble or the best single model becomes the final
  predictor is decided **on validation only** — the test set stays untouched.
- **Test-Time Augmentation (TTA)** — at evaluation, average each image's
  prediction with a few MRI-safe augmented copies. Small, free robustness gain.
- **Bootstrap CIs** — report 95% confidence intervals on the test accuracy and
  Macro-F1 instead of a single point estimate.

In [ ]:
# === Ensemble / TTA / Bootstrap helpers (reuse already-trained models) =======
# Each transfer model applies its OWN preprocess_input internally, so we feed raw
# [0,255] images straight in (TTA augmentations are applied on raw images too).

def make_predict_fn(models):
    """imgs -> averaged softmax probs. Accepts a single model or a list (ensemble)."""
    models = models if isinstance(models, (list, tuple)) else [models]
    def _fn(imgs):
        return np.mean([m.predict(imgs, verbose=0) for m in models], axis=0)
    return _fn

# MRI-safe TTA transforms (same gentle family used during training)
_tta_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
], name='tta_aug')

def predict_dataset(predict_fn, ds, tta_passes=0):
    """Run predict_fn over a dataset. tta_passes=0 -> plain; >0 averages the
    original prediction with that many augmented passes (test-time augmentation)."""
    all_true, all_prob = [], []
    for imgs, labels in ds:
        p = predict_fn(imgs)                       # identity pass
        for _ in range(tta_passes):
            p = p + predict_fn(_tta_aug(imgs, training=True))
        all_prob.append(p / (1 + tta_passes))
        all_true.append(labels.numpy())
    return np.concatenate(all_true), np.concatenate(all_prob)

def bootstrap_ci(y_true, y_pred, n_boot=2000, alpha=0.05):
    """95% bootstrap CI for accuracy and Macro-F1 over the given predictions."""
    rng = np.random.default_rng(SEED)
    n = len(y_true); accs, f1s = [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        accs.append(accuracy_score(y_true[idx], y_pred[idx]))
        f1s.append(f1_score(y_true[idx], y_pred[idx], average='macro'))
    pct = lambda v: (np.percentile(v, 100*alpha/2), np.percentile(v, 100*(1-alpha/2)))
    return pct(accs), pct(f1s)

In [ ]:
# Build the ensemble and choose the final predictor on VALIDATION only.
ft_models   = [finetuned[n][0] for n in TRANSFER_BACKBONES]
ensemble_fn = make_predict_fn(ft_models)

yv, pv  = predict_dataset(ensemble_fn, val_ds, tta_passes=0)
ens_val = compute_metrics(yv, np.argmax(pv, axis=1), pv)
best_single_val_f1 = float(ft_df.iloc[0]['Val Macro-F1'])

print(f'Best single ({best_name})  val Macro-F1 : {best_single_val_f1:.4f}')
print(f'Ensemble (3 fine-tuned)    val Macro-F1 : {ens_val["macro_f1"]:.4f}')

if ens_val['macro_f1'] >= best_single_val_f1:
    final_predict_fn, final_name = ensemble_fn, 'Ensemble (3 fine-tuned)'
else:
    final_predict_fn, final_name = make_predict_fn(best_model), f'{best_name} (fine-tuned)'
print(f'\nFinal predictor (chosen on validation): {final_name}')

## 10. Final Evaluation on the Test Set (performed once)

This is the **only** place the test set is used. The final predictor (best single
model vs. the 3-model ensemble) and the use of TTA were both decided earlier on
the validation set, so this single evaluation does not peek. Reported with a
confusion matrix, full classification report, per-class recall, one-vs-rest
ROC/AUC, and **95% bootstrap confidence intervals** on accuracy and Macro-F1.

In [ ]:
# === SINGLE, FINAL evaluation on the test set ================================
# The final predictor (single vs ensemble) and TTA were both decided beforehand,
# so the test set is still touched exactly once, right here.
USE_TTA    = True          # set False to disable test-time augmentation
TTA_PASSES = 4 if USE_TTA else 0

print(f'=== FINAL TEST EVALUATION - {final_name}'
      + (f' + TTA x{TTA_PASSES}' if TTA_PASSES else '') + ' ===')

y_true, y_prob = predict_dataset(final_predict_fn, test_ds, tta_passes=TTA_PASSES)
y_pred = np.argmax(y_prob, axis=1)
test_metrics = compute_metrics(y_true, y_pred, y_prob)

print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))

# 95% bootstrap confidence intervals -> honest uncertainty on the headline numbers
(acc_lo, acc_hi), (f1_lo, f1_hi) = bootstrap_ci(y_true, y_pred)
print(f"Accuracy : {test_metrics['accuracy']:.4f}  (95% CI {acc_lo:.4f}-{acc_hi:.4f})")
print(f"Macro-F1 : {test_metrics['macro_f1']:.4f}  (95% CI {f1_lo:.4f}-{f1_hi:.4f})")
print(f"Macro AUC: {test_metrics['macro_auc']:.4f}")

plot_confusion_matrix(confusion_matrix(y_true, y_pred), f'{final_name} - Confusion Matrix')
plot_roc(y_true, y_prob, title='Test ROC (one-vs-rest)')

print('\nPer-class recall (clinical sensitivity):')
for c, r in test_metrics['per_class_recall'].items():
    print(f'  {c:12s}: {r:.4f}')

## 11. Explainability — Grad-CAM

Grad-CAM shows *where* the model looks. For a medical model this is essential: it
lets us check the network attends to the tumour region rather than image artefacts.
This is a **qualitative** check, not a performance metric.

In [ ]:
def find_last_conv_layer(base):
    for layer in reversed(base.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
    raise ValueError('No Conv2D layer found in backbone.')

def build_cam_model(model, base, last_conv_name):
    # Re-apply the head layers (everything after the backbone) onto base.output so
    # the predictions are computed *through* the conv features -> gradients flow.
    # This avoids the "Graph disconnected" pitfall of nested pretrained models.
    feat = base.get_layer(last_conv_name).output
    x = base.output
    started = False
    for layer in model.layers:
        if layer.name == base.name:
            started = True
            continue
        if started:
            x = layer(x)
    return tf.keras.Model(base.input, [feat, x])

def make_gradcam_heatmap(preprocessed_img, cam_model, pred_index=None):
    with tf.GradientTape() as tape:
        conv_out, preds = cam_model(preprocessed_img)
        if pred_index is None:
            pred_index = int(tf.argmax(preds[0]))
        class_channel = preds[:, pred_index]
    grads = tape.gradient(class_channel, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.squeeze(conv_out[0] @ pooled[..., tf.newaxis])
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), pred_index

# Build the Grad-CAM model for the fine-tuned winner
best_preprocess = TRANSFER_BACKBONES[best_name][1]
last_conv_name  = find_last_conv_layer(best_base)
cam_model = build_cam_model(best_model, best_base, last_conv_name)
print('Grad-CAM target conv layer:', last_conv_name)

In [ ]:
def show_gradcam(raw_img, true_label):
    img = tf.expand_dims(raw_img, 0)
    pre = best_preprocess(tf.identity(img))
    heatmap, pred_idx = make_gradcam_heatmap(pre, cam_model)
    heatmap = tf.image.resize(heatmap[..., tf.newaxis], IMG_DIMS).numpy().squeeze()
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    ax[0].imshow(raw_img.numpy().astype('uint8'))
    ax[0].set_title(f'True: {CLASSES[true_label]}'); ax[0].axis('off')
    ax[1].imshow(raw_img.numpy().astype('uint8'))
    ax[1].imshow(heatmap, cmap='jet', alpha=0.4)
    ax[1].set_title(f'Grad-CAM — pred: {CLASSES[pred_idx]}'); ax[1].axis('off')
    plt.tight_layout(); plt.show()

# One example from each tumour class (skip 'notumor')
samples = {}
for imgs, labels in test_ds.unbatch():
    l = int(labels)
    if CLASSES[l] != 'notumor' and l not in samples:
        samples[l] = imgs
    if len(samples) >= 3:
        break

for label, img in samples.items():
    show_gradcam(img, label)

## 12. Experiment Summary

In [ ]:
summary = (pd.DataFrame(experiment_log)
           .sort_values('Val Macro-F1', ascending=False)
           .reset_index(drop=True))
display(summary)
print(f"\nFinal predictor: {final_name}")
print(f"Test accuracy : {test_metrics['accuracy']:.4f}")
print(f"Test Macro-F1 : {test_metrics['macro_f1']:.4f}")
print(f"Test Macro AUC: {test_metrics['macro_auc']:.4f}")

## 13. Discussion, Limitations & Future Work

**Discussion.** Transfer learning + fine-tuning clearly outperforms the from-scratch
baseline. The winning backbone was selected purely on validation Macro-F1, and the
test set was used only once, so the reported test numbers are an honest estimate of
held-out performance. We report per-class recall because, clinically, a missed tumour
(false negative) is far costlier than a false alarm — accuracy alone hides this.

**Limitations (read this before trusting the headline number).**
- **No patient IDs.** This dataset does not expose which slices belong to which
  patient. Near-duplicate slices from the same patient may sit in both train and test,
  which can *inflate* the test score. We therefore treat very high accuracy as an
  upper bound, not a clinical guarantee.
- **Single dataset.** No external/independent dataset was used, so generalisation to
  other scanners, hospitals, or populations is unverified.
- **2D slices.** Real diagnosis uses 3D volumes; classifying isolated 2D slices loses
  spatial context.

**Future work.**
- Patient-level train/test split once patient IDs are available (the single biggest
  fix for the leakage concern).
- Validation on an independent external dataset.
- 3D CNNs on full MRI volumes instead of 2D slices.
- Vision Transformers (ViT) and self-supervised pre-training on medical images.
- Probability calibration (e.g. temperature scaling) for trustworthy confidence scores.